In [21]:
# ============================================
# 2) Imports & Configuration (OPTIMIZED)
# ============================================
# Key changes:
# - Removed: transformers AutoModelForCausalLM (6.5 GB local LLM)
# - Added: groq (cloud API for Whisper STT + LLM inference)
# - Added: vaderSentiment (lightweight sentiment, replaces DistilBERT)

import os
import sys
import math
import json
import wave
import contextlib
import numpy as np
import librosa
import soundfile as sf
import re
import concurrent.futures

from dataclasses import dataclass, asdict, field
from typing import List, Tuple, Dict, Optional
from groq import Groq
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


@dataclass
class QuestionGenerationRequest:
    role: str
    experience: str
    company_type: str
    interview_round: str


In [ ]:
# --------------------------------------------
# Determinism (important for calibration)
# --------------------------------------------
np.random.seed(42)

# --------------------------------------------
# Optional ML Dependencies
# --------------------------------------------

nlp = None

# SpaCy (Linguistic analysis — already fast & lightweight, no change needed)
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    NLP_MODE = "spacy"
except Exception:
    nlp = None
    NLP_MODE = "regex"
    print("[WARN] SpaCy model not found. Falling back to regex-based analysis (lower accuracy).")

# VADER Sentiment (lightweight, <1 MB, <10ms — replaces DistilBERT 66M params)
_vader_analyzer = SentimentIntensityAnalyzer()
print("[SENTIMENT] VADER loaded (<1 MB, <10ms inference)")

# --------------------------------------------
# Groq API Configuration
# --------------------------------------------
# Get free API key at: https://console.groq.com/keys

GROQ_API_KEY = "your_groq_api_key_here"  # Replace with your actual Groq API key
if not GROQ_API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY environment variable not set. "
        "Get a free key at https://console.groq.com/keys"
    )

# Model configuration
GROQ_STT_MODEL = "whisper-large-v3-turbo"       # Speech-to-text (better than medium, 15x faster)
GROQ_LLM_MODEL = "llama-3.3-70b-versatile"      # TCS + Coaching (high quality, replaces 3B)
GROQ_LLM_FAST_MODEL = "llama-3.1-8b-instant"    # Question generation (fastest)

# Shared Groq client (connection-pooled)
_groq_client = Groq(api_key=GROQ_API_KEY)

print(f"[INFO] NLP mode: {NLP_MODE}")
print(f"[INFO] Groq STT model: {GROQ_STT_MODEL}")
print(f"[INFO] Groq LLM model: {GROQ_LLM_MODEL}")
print(f"[INFO] Groq LLM fast model: {GROQ_LLM_FAST_MODEL}")
print("[INFO] All models are cloud-hosted via Groq API (zero local GPU/RAM usage)")

# --------------------------------------------
# Interview Analysis Global Config
# (Authoritative tuning source)
# --------------------------------------------

INTERVIEW_CONFIG = {
    # Speaking pace
    "ideal_wpm_range": (125, 145),
    "acceptable_wpm_range": (145, 160),
    "hard_penalty_wpm": 160,

    # Fillers & hesitation
    "max_safe_fillers_per_min": 3.0,
    "filler_penalty_weight": 1.0,
    "max_filler_bonus": 5,

    # Hedging & confidence
    "hedge_penalty_weight": 2.5,
    "min_confidence_score": 40,

    # Delivery (pace + rhythm + pauses)
    "delivery_penalty_weight": 0.20,

    # Structure-agnostic speech control
    "long_block_penalty": 12,
}

[SENTIMENT] VADER loaded (<1 MB, <10ms inference)
[INFO] NLP mode: spacy
[INFO] Groq STT model: whisper-large-v3-turbo
[INFO] Groq LLM model: llama-3.3-70b-versatile
[INFO] Groq LLM fast model: llama-3.1-8b-instant
[INFO] All models are cloud-hosted via Groq API (zero local GPU/RAM usage)


In [23]:
# ============================================
# 3) Audio Utilities & Pitch Analysis
# ============================================

def load_audio_mono(path: str, sr: int = 16000):
    """Load audio as mono float32 at target sampling rate."""
    audio, orig_sr = librosa.load(path, sr=None, mono=True)
    if orig_sr != sr:
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)
    return audio.astype(np.float32), sr


def analyze_pitch_dynamics(audio: np.ndarray, sr: int) -> Dict:
    """
    Analyze pitch variation to detect monotone delivery.

    Returns:
        {
            'std_semitones': float,
            'voiced_ratio': float,
            'monotone_score': float,   # 0 (expressive) → 1 (very monotone)
            'is_monotone': bool
        }
    """
    try:
        f0, voiced_flag, _ = librosa.pyin(
            audio,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr,
            frame_length=2048
        )

        # Use voiced_flag for accurate voiced coverage
        voiced_f0 = f0[voiced_flag]
        voiced_ratio = float(np.mean(voiced_flag)) if len(voiced_flag) else 0.0

        # Not enough voiced speech → unreliable prosody signal
        if len(voiced_f0) < 10 or voiced_ratio < 0.25:
            return {
                "std_semitones": 0.0,
                "voiced_ratio": voiced_ratio,
                "monotone_score": 0.0,
                "is_monotone": False
            }

        # Convert Hz → semitones (log scale, speaker-independent)
        ref_hz = max(np.mean(voiced_f0), 1e-3)  # numerical safety
        semitones = 12.0 * np.log2(voiced_f0 / ref_hz)
        std_semitones = float(np.std(semitones))

        # Interview-calibrated thresholds
        MONOTONE_CENTER = 2.5   # semitones
        MONOTONE_LIMIT = 1.8

        monotone_score = np.clip(
            (MONOTONE_CENTER - std_semitones) / MONOTONE_CENTER,
            0.0,
            1.0
        )
        is_monotone = std_semitones < MONOTONE_LIMIT

        return {
            "std_semitones": std_semitones,
            "voiced_ratio": voiced_ratio,
            "monotone_score": monotone_score,
            "is_monotone": is_monotone
        }

    except Exception as e:
        print(f"[WARN] Pitch analysis failed: {e}")
        return {
            "std_semitones": 0.0,
            "voiced_ratio": 0.0,
            "monotone_score": 0.0,
            "is_monotone": False
        }


In [24]:
# ============================================
# 4) Transcription (Groq Whisper API)
# ============================================
# OPTIMIZED: Replaced local faster-whisper medium (CPU, 30-90s)
# with Groq Whisper API (whisper-large-v3-turbo, 2-5s)
# - 15-18x faster
# - Better accuracy (large-v3-turbo WER ~5% vs medium WER ~8%)
# - Zero local GPU/CPU load
# - Word-level timestamps preserved for pause/segment analysis

@dataclass
class TranscriptionResult:
    text: str
    segments: List[Dict]
    language: str
    duration: float
    num_segments: int


def transcribe_audio(
    audio_path: str,
    model_size: str = "whisper-large-v3-turbo"
) -> TranscriptionResult:
    """
    Transcribe audio using Groq's hosted Whisper API.
    Groq's LPU hardware transcribes 2 min of audio in ~2-5 seconds.
    """
    print(f"[STT] Transcribing via Groq Whisper ({GROQ_STT_MODEL})...")

    with open(audio_path, "rb") as audio_file:
        response = _groq_client.audio.transcriptions.create(
            file=(os.path.basename(audio_path), audio_file),
            model=GROQ_STT_MODEL,
            response_format="verbose_json",
            timestamp_granularities=["word", "segment"],
            language="en",
        )

    # Extract text
    text = (response.text or "").strip()

    # Build segment list with word-level timestamps
    seg_list = []
    all_words = []

    # Process segments
    raw_segments = getattr(response, "segments", None) or []
    for seg in raw_segments:
        seg_list.append({
            "start": float(seg.get("start", 0)),
            "end": float(seg.get("end", 0)),
            "text": seg.get("text", ""),
            "words": []
        })

    # Process word-level timestamps
    raw_words = getattr(response, "words", None) or []
    for w in raw_words:
        word_entry = {
            "start": float(w.get("start", 0)),
            "end": float(w.get("end", 0)),
            "word": w.get("word", "")
        }
        all_words.append(word_entry)

        # Assign words to their parent segment
        for seg in seg_list:
            if seg["start"] <= word_entry["start"] <= seg["end"]:
                seg["words"].append(word_entry)
                break

    # Calculate effective spoken duration
    if all_words:
        duration = float(all_words[-1]["end"] - all_words[0]["start"])
    else:
        duration = float(getattr(response, "duration", 0) or 0)

    language = getattr(response, "language", "en") or "en"

    print(f"[STT] Done: {len(text.split())} words, {duration:.1f}s duration")

    return TranscriptionResult(
        text=text,
        segments=seg_list,
        language=language,
        duration=duration,
        num_segments=len(seg_list)
    )

In [25]:
# ============================================
# 5) NLP: Context-Aware Signals
# ============================================

FILLERS_SIMPLE = {"um", "uh", "umm", "uhh"}
MULTI_FILLERS = ["you know", "i mean"]

HEDGE_KEYWORDS = {"maybe", "probably", "possibly", "might"}
HEDGE_PHRASES = [
    "i think", "not sure", "kind of", "sort of",
    "i guess", "might be", "i don't remember"
]

OWNERSHIP_VERBS = {"build", "design", "lead", "implement", "create", "manage", "solve", "drive"}
APOLOGIES = ["sorry", "apologize", "i forgot", "i didn't prepare", "excuse me"]

def detect_signals(transcript: str, segments: List[Dict]) -> Dict:
    text_lower = transcript.lower()
    filler_count = 0
    hedge_count = 0
    own_count = 0
    passive_count = 0
    apology_count = 0

    # -------------------------
    # NLP-aware analysis
    # -------------------------
    if nlp:
        doc = nlp(transcript)

        for token in doc:
            t = token.text.lower()

            # ---- Fillers ----
            if t in FILLERS_SIMPLE:
                filler_count += 1

            if t == "like" and token.pos_ == "INTJ":
                filler_count += 1

            # ---- Hedging (token-level only) ----
            if t in HEDGE_KEYWORDS:
                hedge_count += 1

            if t == "think" and token.head.text.lower() == "i":
                hedge_count += 1

            # ---- Ownership (broader patterns) ----
            if token.lemma_ in OWNERSHIP_VERBS:
                if any(tok.text.lower() == "i" for tok in token.subtree):
                    own_count += 1

            # ---- Passive (evasive only) ----
            if token.dep_ == "auxpass":
                # penalize only if no ownership nearby
                if not any(tok.text.lower() == "i" for tok in token.head.subtree):
                    passive_count += 1

        # ---- Phrase-level signals (NO double count) ----
        for f in MULTI_FILLERS:
            filler_count += text_lower.count(f)

        for h in HEDGE_PHRASES:
            hedge_count += text_lower.count(h)

    else:
        # Regex fallback
        filler_count += sum(text_lower.count(f) for f in FILLERS_SIMPLE)
        filler_count += sum(text_lower.count(f) for f in MULTI_FILLERS)
        hedge_count += sum(text_lower.count(h) for h in HEDGE_PHRASES)

    # -------------------------
    # Apologies
    # -------------------------
    apology_count = sum(text_lower.count(a) for a in APOLOGIES)

    # -------------------------
    # Pause & Energy Analysis
    # -------------------------
    long_pauses = 0
    long_speech_blocks = 0

    all_words = []
    for seg in segments:
        all_words.extend(seg.get("words", []))

    for i in range(1, len(all_words)):
        gap = all_words[i]["start"] - all_words[i-1]["end"]
        if gap > 1.2:
            long_pauses += 1

    for seg in segments:
        if (seg["end"] - seg["start"]) > 10.0:
            long_speech_blocks += 1

    # -------------------------
    # Semantic uncertainty
    # -------------------------
    uncertainty_patterns = ["or maybe", "not sure if", "i think it was", "can't remember"]
    hedge_count += sum(text_lower.count(p) for p in uncertainty_patterns)

    return {
        "filler_count": filler_count,
        "hedge_count": hedge_count,
        "own_count": own_count,
        "passive_count": passive_count,
        "apology_count": apology_count,
        "long_pauses": long_pauses,
        "long_speech_blocks": long_speech_blocks
    }


In [26]:
# ============================================
# 6) Advanced Scoring Engine (Calibrated)
# ============================================

@dataclass
class InterviewScore:
    total_score: float
    metrics: Dict
    feedback: List[str]


def calculate_score(
    transcript: str,
    duration: float,
    signals: Dict,
    pitch_data: Dict,
    sentiment_res
) -> InterviewScore:

    score = 100.0
    feedback = []

    duration_min = max(duration / 60.0, 1.0)

    # -------------------------
    # 1. CONFIDENCE (HEDGING)
    # -------------------------
    hedges_per_min = signals["hedge_count"] / duration_min
    if hedges_per_min > 1.0:
        pen = min((hedges_per_min - 1.0) * 4.0, 22.0)
        score -= pen
        feedback.append(
            f"Hedging detected ({hedges_per_min:.1f}/min). Be more decisive."
        )

    if signals["apology_count"] > 0:
        pen = signals["apology_count"] * 4.0
        score -= pen
        feedback.append("Avoid apologizing or underselling yourself.")

    # -------------------------
    # 2. OWNERSHIP vs PASSIVE
    # -------------------------
    own_rate = signals["own_count"] / duration_min
    passive_rate = signals["passive_count"] / duration_min

    if own_rate > passive_rate + 0.5:
        bonus = min((own_rate - passive_rate) * 2.0, 8.0)
        score += bonus
        feedback.append("Good ownership language detected.")
    elif passive_rate > own_rate + 1.0:
        score -= 5.0
        feedback.append("Excessive passive voice. Use active language.")

    # -------------------------
    # 3. DELIVERY
    # -------------------------

    # A) Fillers
    fillers_per_min = signals["filler_count"] / duration_min
    if fillers_per_min > 3.0:
        pen = min((fillers_per_min - 3.0) * 2.0, 15.0)
        score -= pen
        feedback.append(
            f"High filler usage ({fillers_per_min:.1f}/min)."
        )

    # B) Long pauses
    pauses_per_min = signals["long_pauses"] / duration_min
    if pauses_per_min > 2.0:
        pen = min((pauses_per_min - 2.0) * 1.5, 8.0)
        score -= pen
        feedback.append("Frequent long pauses detected.")

    # C) WPM (clearer penalty)
    wpm = (len(transcript.split()) / duration) * 60 if duration > 0 else 0

    if wpm < 115:
        score -= min((115 - wpm) * 0.2, 10.0)
        feedback.append(f"Pace is slow ({wpm:.0f} WPM).")
    elif wpm > 155:
        score -= min((wpm - 155) * 0.4, 15.0)
        feedback.append(f"Pace is fast ({wpm:.0f} WPM). Slow down.")

    # D) Energy consistency
    if signals["long_speech_blocks"] > 0:
        pen = min(signals["long_speech_blocks"] * 4.0, 10.0)
        score -= pen
        feedback.append("Break long explanations with pauses.")

    # -------------------------
    # 4. VOICE MODULATION
    # -------------------------
    monotone_score = pitch_data.get("monotone_score", 0.0)
    if monotone_score > 0.6:
        pen = monotone_score * 8.0
        score -= pen
        feedback.append("Voice sounds monotone. Add variation.")

    # -------------------------
    # 5. SENTIMENT (POLISH ONLY)
    # -------------------------
    if sentiment_res:
        label = sentiment_res[0]["label"]
        conf = sentiment_res[0]["score"]

        if label == "POSITIVE" and conf > 0.9:
            score += 1.5
            feedback.append("Positive tone.")
        elif label == "NEGATIVE" and conf > 0.9:
            score -= 5.0
            feedback.append("Tone sounds uncertain.")

    # -------------------------
    # 6. CONFIDENCE CEILING
    # -------------------------
    # High hedging should cap final score
    if hedges_per_min > 2.0:
        score = min(score, 78.0)

    # -------------------------
    # Final clamp
    # -------------------------
    score = max(0.0, min(100.0, score))
    # Absolute realism cap
    if score > 95:
        score = 95.0


    return InterviewScore(
        total_score=score,
        metrics={
            **signals,
            "wpm": wpm,
            "fillers_per_min": fillers_per_min,
            "monotone_score": monotone_score,
        },
        feedback=feedback,
    )


In [27]:
# ============================================
# 7) Main Runner (OPTIMIZED)
# ============================================
# Changes:
# 1. Parallel pitch analysis + transcription (saves ~3s)
# 2. VADER sentiment (<10ms) replaces DistilBERT (3-5s cold, 260 MB)


def _sample_text_for_sentiment(text: str, max_len: int = 512) -> str:
    """Sample beginning + middle + end for fair sentiment."""
    if len(text) <= max_len:
        return text

    part = max_len // 3
    return (
        text[:part] +
        text[len(text)//2 - part//2 : len(text)//2 + part//2] +
        text[-part:]
    )


def _analyze_sentiment_vader(text: str):
    """
    Analyze sentiment using VADER (rule-based, <10ms).
    Returns in the same format as the old DistilBERT pipeline
    so downstream scoring code needs no changes.
    """
    sample = _sample_text_for_sentiment(text)
    scores = _vader_analyzer.polarity_scores(sample)

    compound = scores["compound"]
    if compound >= 0.05:
        return [{"label": "POSITIVE", "score": min(abs(compound), 1.0)}]
    elif compound <= -0.05:
        return [{"label": "NEGATIVE", "score": min(abs(compound), 1.0)}]
    else:
        return [{"label": "NEUTRAL", "score": 1.0 - abs(compound)}]


def analyze_interview(audio_path: str):
    print(f"--- Analyzing {os.path.basename(audio_path)} ---")

    # -------------------------
    # 1. Load Audio
    # -------------------------
    print("Loading audio...")
    audio, sr = load_audio_mono(audio_path)

    # -------------------------
    # 2. PARALLEL: Pitch Analysis + Transcription
    #    These are independent — run concurrently (saves ~3s)
    # -------------------------
    print("Running pitch analysis and transcription in parallel...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        pitch_future = executor.submit(analyze_pitch_dynamics, audio, sr)
        transcribe_future = executor.submit(transcribe_audio, audio_path)

        pitch_data = pitch_future.result()
        tr = transcribe_future.result()

    if tr is None:
        print("[ERROR] Transcription failed. Whisper returned None.")
        return None

    if not tr.text or not tr.text.strip():
        print("[ERROR] Empty transcription. Audio may be silent or corrupted.")
        return None

    # Use effective spoken duration
    duration = tr.duration if tr.duration > 0 else len(audio) / sr

    # -------------------------
    # 3. Linguistic Signals
    # -------------------------
    print("Analyzing linguistic signals...")
    signals = detect_signals(tr.text, tr.segments)

    # -------------------------
    # 4. Sentiment (VADER — <10ms, replaces DistilBERT 3-5s)
    # -------------------------
    sent_res = _analyze_sentiment_vader(tr.text)

    # -------------------------
    # 5. CS Score
    # -------------------------
    result = calculate_score(
        transcript=tr.text,
        duration=duration,
        signals=signals,
        pitch_data=pitch_data,
        sentiment_res=sent_res
    )

    # -------------------------
    # 6. Report
    # -------------------------
    print("\n=== INTERVIEW REPORT ===")
    print(f"OVERALL SCORE: {result.total_score:.1f} / 100")

    print("\n--- Feedback ---")
    for f in result.feedback:
        print(f"[ ] {f}")

    print("\n--- Detailed Metrics ---")
    for k, v in result.metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.2f}")
        else:
            print(f"{k}: {v}")

    print("\n--- Debug Info ---")
    print(f"NLP mode: {NLP_MODE}")
    print(f"Speech duration (s): {duration:.2f}")
    print(f"Voiced ratio: {pitch_data.get('voiced_ratio', 'NA')}")

    # =========================
    # RETURN VALUES FOR TCS
    # =========================
    return {
        "transcript": tr.text,
        "cs_score": result.total_score,
        "cs_result": result
    }

In [28]:
# ============================================
# TCS Model Configuration (OPTIMIZED)
# ============================================
# BEFORE: Loaded Llama 3.2 3B locally (6.5 GB RAM, 7s cold start)
# AFTER:  Groq API client (zero local resources, instant startup)
#
# The Groq client (_groq_client) was already initialized in the
# configuration cell above. No model loading needed!

print("[LLM] Using Groq API — no local model to load")
print(f"[LLM] TCS/Coaching model: {GROQ_LLM_MODEL} (70B)")
print(f"[LLM] Question model: {GROQ_LLM_FAST_MODEL} (8B)")
print("[LLM] Saved ~6.5 GB RAM and ~7s startup time")

[LLM] Using Groq API — no local model to load
[LLM] TCS/Coaching model: llama-3.3-70b-versatile (70B)
[LLM] Question model: llama-3.1-8b-instant (8B)
[LLM] Saved ~6.5 GB RAM and ~7s startup time


In [29]:
@dataclass
class TechnicalEvaluationResult:
    score: int
    band: str
    verdict: str

    issues: List[str] = field(default_factory=list)
    improvement_points: List[str] = field(default_factory=list)

    conceptual_score: int | None = None
    specificity_score: int | None = None
    confidence_score: int | None = None


In [30]:
from typing import List

def build_tcs_prompt(question: str | List[str], transcript: str) -> str:
    # Normalize question in case a list/array is passed
    if isinstance(question, list):
        question = next(
            (str(q).strip() for q in question if str(q).strip()),
            "Explain your approach to this problem."
        )
    else:
        question = str(question).strip() or "Explain your approach to this problem."

    return f"""
You are a senior technical interviewer conducting a mock interview.

You must evaluate the candidate STRICTLY based on:
1. The interview question provided
2. The candidate’s answer provided below

You have NO access to the candidate’s resume, background, or intent beyond
what is explicitly stated.

Your responsibilities:
1. Determine whether the candidate actually answered the question asked.
2. Evaluate technical correctness ONLY within the scope of the question.
3. Identify inaccuracies, misconceptions, missing fundamentals, or weak
   explanations relative to the question.
4. Provide transcript-grounded coaching feedback to improve correctness,
   relevance, and clarity.

STRICT EVALUATION RULES:
- Judge relevance: Penalize if the answer partially or fully misses the question.
- Judge correctness: Evaluate only what the candidate actually said.
- Do NOT infer unstated knowledge or intentions.
- Do NOT introduce new tools, technologies, metrics, or concepts.
- Do NOT penalize for advanced topics unless the question explicitly requires them.
- Avoid generic interview advice (e.g., “practice more”, “be confident”).

Interview Question:
{question}

Candidate Answer:
{transcript}

SCORING GUIDELINES:
- Score from 0 to 100 based on relevance + technical correctness.
- Use these bands:
  - Excellent: Fully answers the question with correct and clear explanation
  - Good: Answers the question correctly with minor gaps or imprecision
  - Partial: Addresses the question but with notable gaps or confusion
  - Weak: Poor alignment with the question or flawed understanding
  - Poor: Does not answer the question or is mostly incorrect

COACHING REQUIREMENTS:
- Provide at least 4 coaching points.
- EACH coaching point must reference:
  - something said in the answer, OR
  - something clearly missing relative to the question.
- If the answer is strong, focus on improving precision, structure, or depth
  without adding new content.

OUTPUT RULES:
- Start the response with '{{' and end with '}}'.
- Respond in STRICT JSON ONLY.
- Do NOT include markdown, explanations, or extra text.

JSON format:
{{
  "score": <int>,
  "band": "<Excellent|Good|Partial|Weak|Poor>",
  "verdict": "<1–2 sentence technical summary judging alignment with the question>",
  "issues": ["<question-relative technical issues or 'No major technical issues identified'>"],
  "improvement_points": ["<specific, question-grounded coaching points>"]
}}

Return only valid JSON.
"""


In [31]:
# ============================================
# _run_llm via Groq API (OPTIMIZED)
# ============================================
# BEFORE: Local Llama 3B on MPS, 15-45s per call, frequent JSON failures
# AFTER:  Groq API with 70B model, 1-3s per call, JSON mode = guaranteed valid output

def _run_llm(prompt: str, max_new_tokens: int = 1600) -> dict:
    """
    Run a prompt through Groq's LLM API and return parsed JSON.
    Uses JSON mode for guaranteed valid output.
    """
    print(f"[LLM] Sending prompt to Groq ({GROQ_LLM_MODEL})...")

    response = _groq_client.chat.completions.create(
        model=GROQ_LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a precise evaluation assistant. "
                    "Always respond with valid JSON only. "
                    "Do not include any text outside the JSON object."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=max_new_tokens,
        temperature=0,
        response_format={"type": "json_object"},
    )

    raw_text = response.choices[0].message.content.strip()
    print(f"[LLM] Response received ({len(raw_text)} chars)")

    try:
        result = json.loads(raw_text)
    except json.JSONDecodeError as e:
        # Fallback: defensive extraction (shouldn't happen with JSON mode)
        print(f"[LLM][WARN] JSON parse failed: {e}")
        parsed_objects = extract_valid_json_objects(raw_text)
        if parsed_objects:
            result = parsed_objects[-1]
        else:
            raise RuntimeError(
                "LLM output could not be parsed into valid JSON.\n"
                "Raw output:\n" + raw_text[-1000:]
            )

    print(f"[LLM] Parsed JSON keys: {list(result.keys())}")
    return result

In [32]:
import json

def extract_valid_json_objects(text: str) -> list[dict]:
    """
    Extracts ALL fully balanced JSON objects from text
    and returns them as parsed dicts.
    """
    results = []
    stack = []
    start = None

    for i, ch in enumerate(text):
        if ch == "{":
            if not stack:
                start = i
            stack.append("{")

        elif ch == "}":
            if stack:
                stack.pop()
                if not stack and start is not None:
                    candidate = text[start:i+1]
                    try:
                        results.append(json.loads(candidate))
                    except json.JSONDecodeError:
                        pass
                    start = None

    return results


In [33]:
def bucket_tcs(score: int) -> str:
    if score >= 85:
        return "Excellent"
    elif score >= 75:
        return "Good"
    elif score >= 60:
        return "Partial"
    elif score >= 35:
        return "Weak"
    else:
        return "Poor"


In [34]:
# ============================================
# run_llm_question via Groq API (OPTIMIZED)
# ============================================
# BEFORE: Local Llama 3B, 10-30s per call, complex JSON repair logic
# AFTER:  Groq API with 8B-instant model, 0.5-1.5s, JSON mode

def run_llm_question(prompt: str, max_new_tokens: int = 512) -> Dict:
    """
    Generate interview questions via Groq's fast LLM.
    Uses 8B-instant model since question generation is simpler.
    """
    print(f"[Q-GEN] Generating questions via Groq ({GROQ_LLM_FAST_MODEL})...")

    response = _groq_client.chat.completions.create(
        model=GROQ_LLM_FAST_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional interviewer. "
                    "Always respond with valid JSON only. "
                    "Do not include any text outside the JSON object."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=max_new_tokens,
        temperature=0.7,  # Slight creativity for diverse questions
        response_format={"type": "json_object"},
    )

    raw_text = response.choices[0].message.content.strip()
    print(f"[Q-GEN] Response received ({len(raw_text)} chars)")

    try:
        result = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"[Q-GEN][WARN] JSON parse failed: {e}")
        parsed_objects = extract_valid_json_objects(raw_text)
        if parsed_objects:
            result = parsed_objects[-1]
        else:
            raise RuntimeError(
                "Question LLM returned invalid JSON.\nRaw output:\n" + raw_text
            )

    print(f"[Q-GEN] Parsed JSON keys: {list(result.keys())}")
    return result

In [35]:
def build_question_generation_prompt(req: QuestionGenerationRequest) -> str:
    return f"""
You are a professional interviewer.

Interview Context:
- Role: {req.role}
- Experience Level: {req.experience}
- Company Type: {req.company_type}
- Interview Round: {req.interview_round}

QUESTION COUNT RULES:
- Ask questions based on the Experience Level ,Role,Company Type , Interview Round.
- HR Round: exactly 6 questions
- Technical Round: exactly 8 questions
- DSA Round: exactly 7 questions
- Coding Round: exactly 5 questions
- Communication Round: exactly 5 questions


CRITICAL OUTPUT CONTRACT:
- Output ONLY one valid JSON object.
- Do NOT add any text before or after the JSON.
- Do NOT add explanations, notes, labels, or headings.
- Stop generating immediately after the final closing brace.
- If any extra text is added, the output is INVALID.

MANDATORY JSON FORMAT:
{{
  "questions": [
    "question_1",
    "question_2",
    "question_3",
    "question_4",
    "question_5"
  ]
}}

IMPORTANT:
- The number of questions in the array MUST exactly match the rule for the selected Interview Round.

Return the JSON now and stop.
""".strip()

In [36]:
from typing import List, Dict

def generate_interview_questions(req: QuestionGenerationRequest) -> List[str]:
    response: Dict = run_llm_question(
        build_question_generation_prompt(req),
        max_new_tokens=512
    )

    if "questions" not in response:
        raise RuntimeError("LLM response missing 'questions' field")

    if not isinstance(response["questions"], list):
        raise RuntimeError("'questions' must be a list")

    questions = [str(q).strip() for q in response["questions"] if str(q).strip()]

    expected_counts = {
        "HR": 6,
        "Technical": 8,
        "DSA": 7,
        "Coding": 5,
        "Communication": 5,
    }
    expected = expected_counts.get(req.interview_round)

    if expected:
        if len(questions) > expected:
            questions = questions[:expected]
        elif len(questions) < expected:
            raise RuntimeError(
                f"Expected {expected} questions for {req.interview_round} round, got {len(questions)}. Raw: {response}"
            )

    return questions


def generate_interview_question(req: QuestionGenerationRequest) -> str:
    questions = generate_interview_questions(req)
    if not questions:
        raise RuntimeError("No questions returned by LLM")
    return questions[0]

In [51]:
req = QuestionGenerationRequest(
    role="Software Development Engineer",
    experience="Fresher",
    company_type="Service-Based",
    interview_round="HR"
)

questions = generate_interview_questions(req)

print("Generated Questions:")
for i, q in enumerate(questions, 1):
    print(f"{i}. {q}")


[Q-GEN] Generating questions via Groq (llama-3.1-8b-instant)...
[Q-GEN] Response received (551 chars)
[Q-GEN] Parsed JSON keys: ['questions']
Generated Questions:
1. What are your long-term career goals in the software development industry?
2. How do you handle stress and pressure in a fast-paced work environment?
3. Can you tell us about a time when you had to work with a difficult team member or customer?
4. Why do you want to work for our company, and what do you know about our services?
5. How do you stay current with industry trends and developments?
6. Can you describe a project you worked on in the past that you are particularly proud of?


In [38]:
# req = QuestionGenerationRequest(
#     role="Marketing Manager",
#     experience="1-3 years",
#     company_type="Product-Based",
#     interview_round="Technical Interview"
# )

# questions = generate_interview_questions(req)

# print("Generated Questions:")
# for i, q in enumerate(questions, 1):
#     print(f"{i}. {q}")


In [39]:
print(questions)

['What motivated you to apply for this Software Development Engineer role at our service-based company?', 'Can you walk us through your academic background and how it has prepared you for this role?', 'How do you handle stress and pressure in a fast-paced software development environment?', 'Tell us about a time when you overcame a difficult challenge in your academic or personal life.', 'How do you stay updated with the latest trends and technologies in software development?', "Can you describe a project you worked on in college or university that you're particularly proud of?"]


In [40]:
from typing import List

def run_tcs_llm(
    transcript: str,
    question: str | List[str] | None = None
) -> dict:
    print("[TCS] Running technical correctness evaluation...")

    # Safe default question
    if question is None:
        question = "Explain your approach to this problem."

    return _run_llm(
        build_tcs_prompt(question, transcript),
        max_new_tokens=1600
    )


In [41]:
def compute_tcs(transcript: str, question: str | List[str] | None = None) -> TechnicalEvaluationResult:
    raw = run_tcs_llm(transcript, question)

    # ---- Score ----
    if "score" not in raw:
        raise RuntimeError(f"TCS output missing 'score'. Raw response: {raw}")

    score = int(raw["score"])
    score = max(0, min(score, 100))

    band = bucket_tcs(score)
    verdict = raw.get("verdict", "").strip()

    # ---- Issues ----
    issues = raw.get("issues", [])
    if not isinstance(issues, list) or not issues:
        issues = ["No major technical issues identified."]

    # ---- Coaching points ----
    improvement_points = raw.get("improvement_points", [])
    if not isinstance(improvement_points, list) or not improvement_points:
        improvement_points = [
            "Improve clarity and specificity while explaining technical decisions."
        ]
    
    print(question)

    return TechnicalEvaluationResult(
        score=score,
        band=band,
        verdict=verdict,
        issues=issues,
        improvement_points=improvement_points
    )

In [42]:
def combine_cs_tcs(cs_score: float, tcs: TechnicalEvaluationResult) -> float:
    # ---- Weighted fusion (technical slightly dominant) ----
    final_score = 0.6 * cs_score + 0.4 * tcs.score

    # ---- Interview realism constraints ----
    if tcs.band == "Poor":
        final_score = min(final_score, 45.0)
    elif tcs.band == "Weak":
        final_score = min(final_score, 60.0)
    elif tcs.band == "Partial":
        final_score = min(final_score, 82.0)  # slightly stricter than 85

    # ---- Absolute realism bounds ----
    final_score = min(final_score, max(cs_score, tcs.score))
    final_score = min(final_score, 95.0)
    final_score = max(final_score, 0.0)

    return round(final_score, 1)


In [43]:
import re

def pretty_print_transcript(text: str, line_width: int = 100):
    words = text.split()
    line = []
    for w in words:
        line.append(w)
        if sum(len(x) + 1 for x in line) >= line_width:
            print(" ".join(line))
            line = []
    if line:
        print(" ".join(line))


In [44]:
from typing import List

def build_placement_coaching_prompt(question: str | List[str], transcript: str) -> str:
    if isinstance(question, list):
        question = next(
            (str(q).strip() for q in question if str(q).strip()),
            "Explain your approach to this problem."
        )
    else:
        question = str(question).strip() or "Explain your approach to this problem."

    return f"""
You are a senior placement officer reviewing a mock interview response.

You must evaluate the candidate STRICTLY based on:
- The interview transcript provided below
- Evidence explicitly present in the transcript

You have NO access to the candidate’s resume, background, or intent beyond
what is stated in the transcript.

YOUR RESPONSIBILITIES:
1. Identify concrete strengths demonstrated in the response.
2. Identify placement-relevant weaknesses or gaps visible in the response.
3. Provide focused coaching advice to improve placement readiness.

EVALUATION RULES:
- Base every point directly on the transcript.
- Do NOT invent skills, experience, or achievements.
- Do NOT add tools, technologies, or concepts not mentioned.
- Avoid generic advice (e.g., “practice more”, “be confident”).
- If evidence is limited, infer conservatively from what is missing.

MANDATORY OUTPUT REQUIREMENTS:
- "standout_strengths" MUST contain **3 to 4 distinct items**
- "top_improvements" MUST contain **3 to 4 distinct items**
- "current_gaps" MUST contain **at least 2 items**
- "actionable_improvements" MUST contain **at least 2 items**
- "placement_focus" MUST contain **at least 2 items**
- Each item must be concise and transcript-grounded

OUTPUT CONSTRAINTS:
- Return EXACTLY ONE JSON object.
- Start the response with '{{' and end with '}}'.
- Output STRICT JSON only (no text, no markdown).
- All values MUST be arrays of strings.
- Keep each point short and specific (no long paragraphs).

Interview Transcript:
{transcript}

JSON format (FOLLOW EXACTLY):

{{
  "standout_strengths": [
    "<strength 1>",
    "<strength 2>",
    "<strength 3>",
    "<optional strength 4>"
  ],
  "top_improvements": [
    "<improvement 1>",
    "<improvement 2>",
    "<improvement 3>",
    "<optional improvement 4>"
  ],
  "placement_coaching": {{
    "current_gaps": [
      "<gap 1>",
      "<gap 2>"
    ],
    "actionable_improvements": [
      "<actionable advice 1>",
      "<actionable advice 2>"
    ],
    "placement_focus": [
      "<focus area 1>",
      "<focus area 2>"
    ]
  }}
}}

Return only valid JSON.
"""


In [45]:
from typing import List

def run_placement_coaching_llm(
    transcript: str,
    question: str | List[str] | None = None
) -> dict:
    print("[COACH] Running placement coaching...")

    prompt = build_placement_coaching_prompt(question, transcript)

    try:
        # _run_llm MUST return a valid dict
        output = _run_llm(prompt, max_new_tokens=800)

        if not isinstance(output, dict):
            raise RuntimeError("LLM output is not a JSON object")

        return output

    except Exception as e:
        print("[COACH][WARN] Placement coaching failed. Using hard fallback.")
        print(str(e))

        return {
            "standout_strengths": [
                "Participated actively in the interview"
            ],
            "top_improvements": [
                "Improve clarity and confidence in explanations"
            ],
            "placement_coaching": {
                "current_gaps": [
                    "Responses lack structured depth"
                ],
                "actionable_improvements": [
                    "Practice explaining answers step-by-step"
                ],
                "placement_focus": [
                    "Communication clarity and interview readiness"
                ]
            }
        }


In [46]:
from typing import List

def generate_placement_feedback(
    transcript: str,
    question: str | List[str] | None = None
) -> dict:

    raw = run_placement_coaching_llm(transcript, question)

    # ---- Safe list extraction (NON-DESTRUCTIVE) ----
    def ensure_list(value, fallback):
        if isinstance(value, list) and len(value) > 0:
            return value
        return [fallback]

    standout_strengths = ensure_list(
        raw.get("standout_strengths"),
        "Shows basic engagement during the interview"
    )

    top_improvements = ensure_list(
        raw.get("top_improvements"),
        "Needs more structured and confident explanations"
    )

    # ---- Normalize placement coaching ----
    placement_raw = raw.get("placement_coaching", {})

    placement = {
        "current_gaps": ensure_list(
            placement_raw.get("current_gaps"),
            "Lacks depth or clarity in some responses"
        ),
        "actionable_improvements": ensure_list(
            placement_raw.get("actionable_improvements"),
            "Practice explaining answers step-by-step with examples"
        ),
        "placement_focus": ensure_list(
            placement_raw.get("placement_focus"),
            "Focus on communication clarity and interview readiness"
        ),
    }

    return {
        "standout_strengths": standout_strengths,
        "top_improvements": top_improvements,
        "placement_coaching": placement
    }


In [48]:
# ============================
# CELL 1: COMMUNICATION SCORE
# ============================

audio_path = "outputs/testing_audio.wav"

cs_out = analyze_interview(audio_path)

if cs_out is None:
    raise RuntimeError("CS analysis failed.")

# ---- Extract only what TCS needs ----
transcript = cs_out["transcript"]
cs_score = cs_out["cs_score"]

print("=== COMMUNICATION SCORE (CS) ===")
print(f"CS Score      : {cs_score}")
print("\n--- TRANSCRIPT ---")
pretty_print_transcript(transcript)
print("--- END TRANSCRIPT ---")


--- Analyzing testing_audio.wav ---
Loading audio...
Running pitch analysis and transcription in parallel...
[STT] Transcribing via Groq Whisper (whisper-large-v3-turbo)...
[STT] Done: 270 words, 115.7s duration
Analyzing linguistic signals...

=== INTERVIEW REPORT ===
OVERALL SCORE: 94.5 / 100

--- Feedback ---
[ ] Good ownership language detected.
[ ] Break long explanations with pauses.
[ ] Positive tone.

--- Detailed Metrics ---
filler_count: 0
hedge_count: 0
own_count: 1
passive_count: 0
apology_count: 0
long_pauses: 0
long_speech_blocks: 2
wpm: 140.04
fillers_per_min: 0.00
monotone_score: 0.00

--- Debug Info ---
NLP mode: spacy
Speech duration (s): 115.68
Voiced ratio: 0.44386351128233353
=== COMMUNICATION SCORE (CS) ===
CS Score      : 94.53734439834025

--- TRANSCRIPT ---
Hello, I am Sugan Kumar. I am currently pursuing a bachelor's degree in computer science engineering
with a specialization in artificial intelligence and machine learning. Through my coursework, I have
built

In [49]:
# =========================================
# CELL 2: TCS + FINAL AGGREGATION
# =========================================

# ---- Run Technical Correctness Scoring (SINGLE LLM CALL) ----
tcs = compute_tcs(transcript ,questions)

# ---- Combine CS + TCS ----
final_score = combine_cs_tcs(cs_score, tcs)

# ---- Pretty Output ----
print("\n" + "=" * 45)
print("TECHNICAL CORRECTNESS (TCS)")
print("=" * 45)
print(f"Score : {tcs.score}")
print(f"Band  : {tcs.band}")

print("\nVerdict:")
print(tcs.verdict)

print("\nIssues Identified:")
for issue in tcs.issues:
    print(f"- {issue}")

print("\n" + "=" * 45)
print("COACHING FEEDBACK")
print("=" * 45)
for point in tcs.improvement_points:
    print(f"- {point}")

print("\n" + "=" * 45)
print("FINAL INTERVIEW SCORE")
print("=" * 45)
print(final_score)


[TCS] Running technical correctness evaluation...
[LLM] Sending prompt to Groq (llama-3.3-70b-versatile)...
[LLM] Response received (1798 chars)
[LLM] Parsed JSON keys: ['score', 'band', 'verdict', 'issues', 'improvement_points']
['What motivated you to apply for this Software Development Engineer role at our service-based company?', 'Can you walk us through your academic background and how it has prepared you for this role?', 'How do you handle stress and pressure in a fast-paced software development environment?', 'Tell us about a time when you overcame a difficult challenge in your academic or personal life.', 'How do you stay updated with the latest trends and technologies in software development?', "Can you describe a project you worked on in college or university that you're particularly proud of?"]

TECHNICAL CORRECTNESS (TCS)
Score : 80
Band  : Good

Verdict:
The candidate provided a clear and relevant answer to the question, highlighting their motivation for applying to the So

In [50]:
# ---- Placement Coaching ----
coaching = generate_placement_feedback(transcript, questions)

print("\n" + "=" * 45)
print("RAW COACHING OBJECT")
print("=" * 45)
print(coaching)

# ---- Standout Strengths ----
print("\n" + "=" * 45)
print("STANDOUT STRENGTHS")
print("=" * 45)
for s in coaching["standout_strengths"]:
    print(f"- {s}")

# ---- Top Improvements ----
print("\n" + "=" * 45)
print("TOP IMPROVEMENTS")
print("=" * 45)
for i in coaching["top_improvements"]:
    print(f"- {i}")

# ---- Placement Coaching Insights ----
placement = coaching["placement_coaching"]

print("\n" + "=" * 45)
print("PLACEMENT COACHING INSIGHTS")
print("=" * 45)

print("\nWhere the candidate currently lags:")
for g in placement["current_gaps"]:
    print(f"- {g}")

print("\nWhat should be improved next:")
for a in placement["actionable_improvements"]:
    print(f"- {a}")

print("\nAreas to focus for placements:")
for f in placement["placement_focus"]:
    print(f"- {f}")


[COACH] Running placement coaching...
[LLM] Sending prompt to Groq (llama-3.3-70b-versatile)...
[LLM] Response received (1209 chars)
[LLM] Parsed JSON keys: ['standout_strengths', 'top_improvements', 'placement_coaching']

RAW COACHING OBJECT
{'standout_strengths': ['Strong foundation in core computer science subjects', 'Practical experience with front-end development and problem solving', 'Ability to adapt quickly and continuous learning mindset', 'Experience with relevant tools and frameworks like React and Git'], 'top_improvements': ['Providing specific examples of projects and their impact', 'Discussing experience with collaboration and teamwork', 'Highlighting understanding of industry needs and trends', 'Showcasing ability to work with scalable, high-quality solutions'], 'placement_coaching': {'current_gaps': ['Lack of specific examples of problem-solving experiences', 'Limited discussion of collaboration and teamwork'], 'actionable_improvements': ['Prepare to discuss specific pr